In [3]:
# imports
import pandas as pd
import requests
import zipfile
import io
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset

For this example we will be using the Concrete Compressive Strength Dataset: https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength

In [4]:
# URL to the concrete+compressive+strength.zip file on the UCI archive
concrete_zip_url = "https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip"

# Download the zip file
response = requests.get(concrete_zip_url)
response.raise_for_status() # Raise an exception for bad status codes

# Read the zip file from memory
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # Print all file names in the zip archive to identify the correct one

    # Use the correct Excel file name identified from z.namelist()
    with z.open('Concrete_Data.xls') as excel_file:
        concrete_df = pd.read_excel(excel_file)

# create X and y, scale data, create train and test
target_column_name = concrete_df.columns[-1]
y = concrete_df[target_column_name]
X = concrete_df.drop(target_column_name, axis=1)

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X).astype(np.float32))
y = pd.DataFrame(y.astype(np.float32))

# train / val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=499)
# NOTE: I haven't used a test set in this example because it's all about learning!

display(X_train.head())
display(y_train.head())

,0,1,2,3,4,5,6,7
316,-0.281032,-0.856886,0.715275,-1.659688,1.029528,0.425670,1.574577,-0.279733
554,-0.411326,0.984546,-0.847132,0.193657,-1.038944,0.870881,-0.490150,-0.612331
663,-1.418445,1.462298,-0.847132,0.488805,-1.038944,-0.585704,0.818867,-0.279733
881,-1.226977,0.824522,0.919448,-0.167080,0.300957,0.374201,-1.055435,-0.279733
774,0.965325,-0.856886,-0.847132,0.207711,-1.038944,1.776742,0.130042,-0.612331


,"Concrete compressive strength(MPa, megapascals)"
316,33.942902
554,15.691094
663,27.874825
881,25.558876
774,11.465986


In [5]:
class FCNN(nn.Module):
  def __init__(self, input_size):
    super(FCNN, self).__init__()
    self.fc1 = nn.Linear(input_size, 32)
    self.fc2 = nn.Linear(32, 16)
    self.fc3 = nn.Linear(16, 1)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    return self.fc3(x)

In [6]:
# Initialize the models we defined above
model = FCNN(input_size=X_train.shape[1])

# To use mini-batch GD we need to create a Dataset object
X_train_torch = torch.tensor(X_train.values, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_torch = torch.tensor(X_val.values, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
# Wrap them in a Dataset
train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset = TensorDataset(X_val_torch, y_val_torch)
# Now create a DataLoader for Mini-Batch GD
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define our Loss function and algorithm for optimization (aka our 'Optimizer')
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")


    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

/Users/itzjuztmya/miniconda3/envs/school/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 50/5000 | Batch Loss: 32.1468
Epoch 50/5000 | Batch Loss: 53.0054
Epoch 50/5000 | Batch Loss: 20.8579
Epoch 50/5000 | Batch Loss: 19.4361
Epoch 50/5000 | Batch Loss: 19.7885
Epoch 50/5000 | Batch Loss: 44.0386
Epoch 50/5000 | Batch Loss: 25.4222
Epoch 50/5000 | Batch Loss: 36.8699
Epoch 50/5000 | Batch Loss: 28.2453
Epoch 50/5000 | Batch Loss: 38.0798
Epoch 50/5000 | Batch Loss: 30.8794
Epoch 50/5000 | Batch Loss: 24.9965
Epoch 50/5000 | Batch Loss: 39.0789
Epoch 50/5000 | Batch Loss: 31.3440
Epoch 50/5000 | Batch Loss: 40.4970
Epoch 50/5000 | Batch Loss: 14.7864
Epoch 50/5000 | Batch Loss: 53.3200
Epoch 50/5000 | Batch Loss: 49.1580
Epoch 50/5000 | Batch Loss: 25.9079
Epoch 50/5000 | Batch Loss: 24.3183
Epoch 50/5000 | Batch Loss: 35.7443
Epoch 50/5000 | Batch Loss: 49.2896
Epoch 50/5000 | Batch Loss: 27.1995
Epoch 50/5000 | Batch Loss: 25.7273
Epoch 50/5000 | Batch Loss: 36.1528
Epoch 50/5000 | Batch Loss: 43.7029

Epoch 50: Train Loss: 33.4613 | Val Loss: 43.6684
-------------

In [7]:
# using Adam AND mini-batch GD
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")

    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

Epoch 50/5000 | Batch Loss: 7.6546
Epoch 50/5000 | Batch Loss: 3.0914
Epoch 50/5000 | Batch Loss: 1.3433
Epoch 50/5000 | Batch Loss: 2.8224
Epoch 50/5000 | Batch Loss: 6.2043
Epoch 50/5000 | Batch Loss: 2.7023
Epoch 50/5000 | Batch Loss: 2.8067
Epoch 50/5000 | Batch Loss: 2.9531
Epoch 50/5000 | Batch Loss: 2.3890
Epoch 50/5000 | Batch Loss: 3.4240
Epoch 50/5000 | Batch Loss: 1.7195
Epoch 50/5000 | Batch Loss: 2.0214
Epoch 50/5000 | Batch Loss: 5.4651
Epoch 50/5000 | Batch Loss: 2.1088
Epoch 50/5000 | Batch Loss: 3.1828
Epoch 50/5000 | Batch Loss: 2.4131
Epoch 50/5000 | Batch Loss: 3.1177
Epoch 50/5000 | Batch Loss: 5.8241
Epoch 50/5000 | Batch Loss: 2.5194
Epoch 50/5000 | Batch Loss: 4.3087
Epoch 50/5000 | Batch Loss: 3.6515
Epoch 50/5000 | Batch Loss: 3.8007
Epoch 50/5000 | Batch Loss: 2.0458
Epoch 50/5000 | Batch Loss: 1.8122
Epoch 50/5000 | Batch Loss: 19.9126
Epoch 50/5000 | Batch Loss: 2.1361

Epoch 50: Train Loss: 3.9012 | Val Loss: 21.5144
-----------------------

Epoch 100/5000

In [8]:
# using a learning rate decay with Adam and mini-batch GD
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# LR Decay
# This will multiply the LR by 0.1 every 20 epochs
scheduler = StepLR(optimizer, step_size=20, gamma=0.1)

# training loop
epochs = 5_000

for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train() # Set to training mode
    train_loss = 0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Batch Loss: {loss.item():.4f}")

    # --- VALIDATION PHASE ---
    model.eval() # Set to evaluation mode (turns off Dropout, BatchNorm, etc.)
    val_loss = 0
    with torch.no_grad(): # Disables gradient calculation
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            # Calculate average losses
            avg_train = train_loss / len(train_loader)
            avg_val = val_loss / len(val_loader)

            print()
            print(f"Epoch {epoch + 1}: Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
            print("-----------------------")
            print()

Epoch 50/5000 | Batch Loss: 1.6771
Epoch 50/5000 | Batch Loss: 2.6873
Epoch 50/5000 | Batch Loss: 1.8181
Epoch 50/5000 | Batch Loss: 4.8303
Epoch 50/5000 | Batch Loss: 1.9698
Epoch 50/5000 | Batch Loss: 1.6503
Epoch 50/5000 | Batch Loss: 3.3663
Epoch 50/5000 | Batch Loss: 3.9272
Epoch 50/5000 | Batch Loss: 1.1223
Epoch 50/5000 | Batch Loss: 19.9873
Epoch 50/5000 | Batch Loss: 2.2164
Epoch 50/5000 | Batch Loss: 0.9000
Epoch 50/5000 | Batch Loss: 3.6910
Epoch 50/5000 | Batch Loss: 3.0001
Epoch 50/5000 | Batch Loss: 0.4815
Epoch 50/5000 | Batch Loss: 1.5439
Epoch 50/5000 | Batch Loss: 1.5981
Epoch 50/5000 | Batch Loss: 1.5676
Epoch 50/5000 | Batch Loss: 1.6602
Epoch 50/5000 | Batch Loss: 1.7491
Epoch 50/5000 | Batch Loss: 1.2619
Epoch 50/5000 | Batch Loss: 1.0557
Epoch 50/5000 | Batch Loss: 1.4159
Epoch 50/5000 | Batch Loss: 4.8364
Epoch 50/5000 | Batch Loss: 1.8251
Epoch 50/5000 | Batch Loss: 0.8320

Epoch 50: Train Loss: 2.7950 | Val Loss: 25.0042
-----------------------

Epoch 100/5000

Can you use the above information to create a plot to visualize the learning rate improving? You will need to add a 'standard' GD option also.